In [1]:
import pandas as pd
import numpy as np

In [2]:
recipes = pd.read_csv('../data/recipes_sample.csv')
reviews = pd.read_csv('../data/reviews_sample.csv')

print(recipes.shape)
print(reviews.shape)

(9997, 6)
(136359, 8)


In [3]:
recipe_cols = ['RecipeId', 'Name', 'Description', 'Images', 'RecipeInstructions']
recipes = recipes[recipe_cols]
recipes.head()

,RecipeId,Name,Description,Images,RecipeInstructions
0,62,"Black Bean, Corn, and Tomato Salad","This is easy, delicious, colorful, delicious, ...",character(0),"c(""In a bowl whisk together lemon juice, oil, ..."
1,66,Black Coffee Barbecue Sauce,It's great to know folks like this sauce so mu...,"c(""https://img.sndimg.com/food/image/upload/w_...","c(""Combine all ingredients in a saucepan and s..."
2,76,Alfredo Sauce,This is my son's favorite meal. I make it with...,"c(""https://img.sndimg.com/food/image/upload/w_...","c(""Place butter in microwave safe pot and heat..."
3,129,Champagne Punch,Here is a good punch recipe for any special oc...,"""https://img.sndimg.com/food/image/upload/w_55...","c(""Mix the juice concentrates in punch bowl (d..."
4,153,Amish Friendship Bread and Starter,Many recipes have been posted for the Amish br...,"c(""https://img.sndimg.com/food/image/upload/w_...","c(""Place ONE cup each sugar, milk, and flour i..."


In [4]:
review_cols = ['ReviewId', 'RecipeId', 'AuthorId', 'Rating', 'Review', 'DateSubmitted']
reviews = reviews[review_cols]
reviews.head()

,ReviewId,RecipeId,AuthorId,Rating,Review,DateSubmitted
0,243,810,2312,0,"Good, but I wished they were a bit more moist.",2001-01-02T16:15:26Z
1,517,4296,6406,5,Found this recipie to be delicious and easy to...,2001-03-12T15:27:12Z
2,526,8600,2312,4,"Fancy macaroni and cheese, so you don't have t...",2001-03-13T19:51:36Z
3,700,2886,2312,5,I used one more banana than called for because...,2001-04-18T08:53:27Z
4,791,8806,7802,0,I generally keep unused homemade hot sauce in ...,2001-05-10T10:23:37Z


In [5]:
m = len(reviews['RecipeId'].unique())
n = len(reviews['AuthorId'].unique())
print(m,n)

9997 5000


# Form Matrix

In [6]:
from scipy.sparse import coo_array, csr_array, coo_matrix, csr_matrix

recipe_ids = np.sort(reviews['RecipeId'].unique())
user_ids = np.sort(reviews['AuthorId'].unique())

recipe_id_map = pd.Series(index = recipe_ids, data = np.arange(0,m))
user_id_map = pd.Series(index = user_ids, data = np.arange(0,n))

reviews['new_recipe_id'] = recipe_id_map[reviews['RecipeId']].reset_index(drop = True)
reviews['new_user_id'] = user_id_map[reviews['AuthorId']].reset_index(drop = True)
recipes = recipes[recipes['RecipeId'].isin(recipe_ids)].reset_index(drop = True)
recipes['new_recipe_id'] = recipe_id_map[recipes['RecipeId']].reset_index(drop = True)

matrix = coo_matrix(
    (reviews['Rating'], (reviews['new_user_id'], reviews['new_recipe_id'])),
    shape = (n,m)
).tocsr()

In [7]:
num_users, num_items = matrix.shape
num_interactions = matrix.nnz

print(f"Users: {num_users}")
print(f"Items: {num_items}")
print(f"Interactions: {num_interactions}")
print(f"Sparsity: {num_interactions / (num_users * num_items):.6f}")

Users: 5000
Items: 9997
Interactions: 136359
Sparsity: 0.002728


In [8]:
from scipy.sparse import save_npz

save_npz("../data/interaction_matrix.npz", matrix)